# **MÔ HÌNH CHÍNH -  YOLOV8**

### **1. Thiết lập môi trường và Cấu hình ban đầu**

#### **1.1. Mount Drive và cài thư viện**

In [1]:
from google.colab import drive
drive.mount("/content/drive")

!pip install ultralytics opencv-python pandas pyyaml -q

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 39.0 MB/s eta 0:00:00


#### **1.2. Cấu hình đường dẫn và tham số mô hình (Hyperparameters)**

In [2]:
import os, json, time, shutil, random
from pathlib import Path
from datetime import datetime

import yaml
import pandas as pd
import torch
from ultralytics import YOLO
import zipfile

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
ZIP_PATH = "/content/drive/MyDrive/ObjectDetection/voc_traffic_5classes.zip"

EXTRACT_ROOT = Path("/content")

YOLO_DIR = EXTRACT_ROOT / "yolo"

DATA_YAML = YOLO_DIR / "data.yaml"

OUTPUT_ROOT = Path("/content/app_model_results")
MODEL_NAME = "yolov8s_subset"
RUN_NAME = "yolov8s_subset_train"

CLASSES = ["person", "car", "bus", "bicycle", "motorbike"]

EPOCHS = 30
BATCH_SIZE = 16
IMAGE_SIZE = 640

DEVICE = 0 if torch.cuda.is_available() else "cpu"
DEVICE_NAME = "cuda" if torch.cuda.is_available() else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"

MODEL_RESULT_DIR = OUTPUT_ROOT / MODEL_NAME
MODEL_DIR = MODEL_RESULT_DIR / "model"
CONFIG_DIR = MODEL_RESULT_DIR / "config"
METRICS_DIR = MODEL_RESULT_DIR / "metrics"
PREDICTIONS_DIR = MODEL_RESULT_DIR / "predictions"
SAMPLE_PRED_DIR = PREDICTIONS_DIR / "sample_predictions"
LOGS_DIR = MODEL_RESULT_DIR / "logs"

for d in [
    MODEL_DIR,
    CONFIG_DIR,
    METRICS_DIR,
    PREDICTIONS_DIR,
    SAMPLE_PRED_DIR,
    LOGS_DIR
]:
    d.mkdir(parents=True, exist_ok=True)

TRAINING_LOG_PATH = LOGS_DIR / "training_log.txt"

with open(TRAINING_LOG_PATH, "w", encoding="utf-8") as f:
    f.write("YOLOv8s Full Dataset Training Log\n")
    f.write("=" * 50 + "\n")
    f.write(f"Timestamp: {datetime.now()}\n")
    f.write(f"ZIP_PATH: {ZIP_PATH}\n")
    f.write(f"YOLO_DIR: {YOLO_DIR}\n")
    f.write(f"DATA_YAML: {DATA_YAML}\n")
    f.write(f"GPU: {GPU_NAME}\n")

### **2. Chuẩn bị dữ liệu (Data Preparation)**

#### **2.1. Kiểm tra dataset**

In [4]:
if not YOLO_DIR.exists():
    print("Unzipping dataset...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(EXTRACT_ROOT)
    print("Dataset unzipped.")

if not DATA_YAML.exists():
    raise FileNotFoundError(f"Không tìm thấy DATA_YAML: {DATA_YAML}")

with open(DATA_YAML, "r", encoding="utf-8") as f:
    raw_data_yaml = yaml.safe_load(f)

# sửa path tuyệt đối cho Colab
raw_data_yaml["path"] = str(YOLO_DIR)

print(raw_data_yaml)

Unzipping dataset...
Dataset unzipped.
{'path': '/content/yolo', 'train': 'images/train', 'val': 'images/val', 'test': 'images/test', 'names': {0: 'person', 1: 'car', 2: 'bus', 3: 'bicycle', 4: 'motorbike'}}


#### **2.2. Cập nhật file cấu hình dữ liệu (data.yaml)**

In [5]:
with open(DATA_YAML, "r", encoding="utf-8") as f:
    data_yaml = yaml.safe_load(f)

# Đảm bảo path đúng
data_yaml["path"] = str(DATA_YAML.parent)

# Đảm bảo thứ tự class đúng
data_yaml["names"] = {
    0: "person",
    1: "car",
    2: "bus",
    3: "bicycle",
    4: "motorbike"
}

FIXED_DATA_YAML = CONFIG_DIR / "data.yaml"

with open(FIXED_DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(data_yaml, f, allow_unicode=True, sort_keys=False)

print("Saved fixed data.yaml:", FIXED_DATA_YAML)

Saved fixed data.yaml: /content/app_model_results/yolov8s_subset/config/data.yaml


#### **2.3. Đếm và kiểm tra số lượng ảnh (Train/Val/Test)**

In [6]:
def count_images(split):
    with open(FIXED_DATA_YAML, "r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)

    img_dir = Path(cfg["path"]) / cfg[split]

    total = 0
    for ext in ["*.jpg", "*.jpeg", "*.png"]:
        total += len(list(img_dir.glob(ext)))
    return total

TRAIN_IMAGES = count_images("train")
VAL_IMAGES = count_images("val")
TEST_IMAGES = count_images("test")

print("Train images:", TRAIN_IMAGES)
print("Val images:", VAL_IMAGES)
print("Test images:", TEST_IMAGES)

with open(TRAINING_LOG_PATH, "a", encoding="utf-8") as f:
    f.write(f"Train images: {TRAIN_IMAGES}\n")
    f.write(f"Val images: {VAL_IMAGES}\n")
    f.write(f"Test images: {TEST_IMAGES}\n")

Train images: 17265
Val images: 2157
Test images: 2157


#### **2.4. Lưu config cho training và app**

In [7]:
training_config = {
    "model_name": "YOLOv8s subset",
    "model_type": "One-stage object detector",
    "dataset": "VOC0712 traffic 5 classes subset",
    "dataset_path": str(FIXED_DATA_YAML),
    "classes": CLASSES,
    "num_classes": 5,
    "train_images": TRAIN_IMAGES,
    "val_images": VAL_IMAGES,
    "test_images": TEST_IMAGES,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "image_size": IMAGE_SIZE,
    "learning_rate": "default_ultralytics",
    "weight_decay": "default_ultralytics",
    "optimizer": "auto/default_ultralytics",
    "scheduler": "default_ultralytics",
    "pretrained": True,
    "freeze_backbone": False,
    "device": DEVICE_NAME,
    "gpu": GPU_NAME
}

with open(CONFIG_DIR / "training_config.json", "w", encoding="utf-8") as f:
    json.dump(training_config, f, indent=4, ensure_ascii=False)

In [8]:
class_names = {
    "0": "person",
    "1": "car",
    "2": "bus",
    "3": "bicycle",
    "4": "motorbike"
}

with open(CONFIG_DIR / "class_names.json", "w", encoding="utf-8") as f:
    json.dump(class_names, f, ensure_ascii=False, indent=2)

inference_config = {
    "model_name": "YOLOv8s subset",
    "model_file": "model/best.pt",
    "image_size": IMAGE_SIZE,
    "confidence_threshold": 0.25,
    "iou_threshold": 0.45,
    "classes": CLASSES,
    "input_type": ["image", "video", "webcam"],
    "device": "cuda_or_cpu"
}

with open(CONFIG_DIR / "inference_config.json", "w", encoding="utf-8") as f:
    json.dump(inference_config, f, ensure_ascii=False, indent=2)

### **3. Huấn luyện mô hình (Training)**

#### **3.1. Huấn luyện mô hình YOLOv8s**

In [9]:
model = YOLO("yolov8s.pt")

train_start = time.time()

model.train(
    data=str(FIXED_DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    workers=2,
    project=str(LOGS_DIR),
    name=RUN_NAME,
    patience=10,
    save=True,
    plots=True
)

train_end = time.time()

with open(TRAINING_LOG_PATH, "a", encoding="utf-8") as f:
    f.write(f"Training finished. Total time: {train_end - train_start:.2f} seconds\n")

Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/app_model_results/yolov8s_subset/config/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8s_subset_train, nbs=64, nms=False, opset=None, optimize=False, optimiz

#### **3.2. Copy checkpoint cho app**

In [10]:
RUN_DIR = LOGS_DIR / RUN_NAME

YOLO_BEST = RUN_DIR / "weights" / "best.pt"
YOLO_LAST = RUN_DIR / "weights" / "last.pt"

if not YOLO_BEST.exists():
    raise FileNotFoundError(f"Missing best checkpoint: {YOLO_BEST}")

shutil.copy(YOLO_BEST, MODEL_DIR / "best.pt")

if YOLO_LAST.exists():
    shutil.copy(YOLO_LAST, MODEL_DIR / "last.pt")

BEST_MODEL_PATH = MODEL_DIR / "best.pt"

print("Saved best model:", BEST_MODEL_PATH)

Saved best model: /content/app_model_results/yolov8s_subset/model/best.pt


#### **3.3. Trích xuất lịch sử huấn luyện (Training History)**

In [11]:
results_csv = RUN_DIR / "results.csv"

if results_csv.exists():
    shutil.copy(results_csv, LOGS_DIR / "results.csv")
    print("Copied results.csv:", LOGS_DIR / "results.csv")
else:
    print("results.csv not found:", results_csv)

Copied results.csv: /content/app_model_results/yolov8s_subset/logs/results.csv


### **4. Đánh giá mô hình (Evaluation)**

#### **4.1. Đánh giá trên test set**

In [12]:
model = YOLO(str(BEST_MODEL_PATH))

eval_results = model.val(
    data=str(FIXED_DATA_YAML),
    split="test",
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    plots=True,
    project=str(LOGS_DIR),
    name="test_evaluation"
)

mAP50 = float(eval_results.box.map50)
mAP5095 = float(eval_results.box.map)
precision = float(eval_results.box.mp)
recall = float(eval_results.box.mr)

total_parameters = sum(p.numel() for p in model.model.parameters())
trainable_parameters = sum(p.numel() for p in model.model.parameters() if p.requires_grad)
model_size_MB = BEST_MODEL_PATH.stat().st_size / (1024 * 1024)

print("mAP@0.5:", mAP50)
print("mAP@0.5:0.95:", mAP5095)
print("Precision:", precision)
print("Recall:", recall)

Ultralytics 8.4.51 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 11,127,519 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 32.4±12.8 MB/s, size: 99.2 KB)
val: Scanning /content/yolo/labels/test... 2157 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2157/2157 475.1it/s 4.5s
val: New cache created: /content/yolo/labels/test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 135/135 5.1it/s 26.6s
                   all       2157       6069      0.853      0.751      0.846      0.636
                person       1625       4084      0.864      0.745      0.858      0.602
                   car        589       1226      0.836      0.792      0.873      0.657
                   bus        147        203      0.825      0.767      0.835      0.701
               bicycle        194        296      0.897      0.666      0.823      0.59

#### **4.2. Chỉ số đánh giá chi tiết theo từng lớp (Per-class Metrics)**

In [13]:
class_maps = eval_results.box.maps

per_class_rows = []

for i, cls in enumerate(CLASSES):
    per_class_rows.append({
        "class": cls,
        "mAP": float(class_maps[i]) if i < len(class_maps) else None,
        "precision": None,
        "recall": None,
        "num_objects": None
    })

pd.DataFrame(per_class_rows).to_csv(
    METRICS_DIR / "per_class_metrics.csv",
    index=False
)

print("Saved per-class metrics.")

Saved per-class metrics.


### **5. Suy luận và Dự đoán (Inference & Prediction)**

#### **5.1. Đo tốc độ suy luận (Inference Speed & FPS)**

In [14]:
with open(FIXED_DATA_YAML, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

test_img_dir = Path(cfg["path"]) / cfg["test"]

test_images = []
for ext in ["*.jpg", "*.jpeg", "*.png"]:
    test_images.extend(list(test_img_dir.glob(ext)))

speed_sample = test_images[:min(200, len(test_images))]

if len(speed_sample) == 0:
    raise ValueError("Không có ảnh test để đo inference speed.")

start = time.time()
_ = model.predict(
    source=[str(p) for p in speed_sample],
    imgsz=IMAGE_SIZE,
    conf=0.25,
    iou=0.45,
    device=DEVICE,
    verbose=False
)
end = time.time()

avg_inference_time = (end - start) / len(speed_sample)
fps = 1 / avg_inference_time

print("Avg inference time:", avg_inference_time)
print("FPS:", fps)

Avg inference time: 0.021818588972091674
FPS: 45.83247804333762


#### **5.2. Dự đoán thử nghiệm và trực quan hóa (trên 10 ảnh mẫu)**

In [15]:
random.seed(42)

sample_images = random.sample(test_images, min(10, len(test_images)))

prediction_rows = []

pred_results = model.predict(
    source=[str(p) for p in sample_images],
    conf=0.25,
    iou=0.45,
    imgsz=IMAGE_SIZE,
    device=DEVICE,
    save=True,
    project=str(PREDICTIONS_DIR),
    name="sample_predictions_tmp",
    verbose=False
)

tmp_pred_dir = PREDICTIONS_DIR / "sample_predictions_tmp"

for img_path, result in zip(sample_images, pred_results):
    src_pred_img = tmp_pred_dir / img_path.name
    dst_pred_img = SAMPLE_PRED_DIR / img_path.name

    if src_pred_img.exists():
        shutil.copy(src_pred_img, dst_pred_img)

    for box in result.boxes:
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        x1, y1, x2, y2 = box.xyxy[0].tolist()

        prediction_rows.append({
            "image_name": img_path.name,
            "class": CLASSES[cls_id],
            "confidence": conf,
            "bbox_xmin": x1,
            "bbox_ymin": y1,
            "bbox_xmax": x2,
            "bbox_ymax": y2
        })

pd.DataFrame(prediction_rows).to_csv(
    PREDICTIONS_DIR / "prediction_examples.csv",
    index=False
)

if tmp_pred_dir.exists():
    shutil.rmtree(tmp_pred_dir)

print("Saved sample predictions.")

Results saved to /content/app_model_results/yolov8s_subset/predictions/sample_predictions_tmp
Saved sample predictions.


### **6. Tổng hợp kết quả và Lưu trữ**

#### **6.1. Lưu metrics tổng quát**

In [16]:
evaluation_metrics = {
    "model": "YOLOv8s subset",
    "dataset": "VOC0712 traffic 5 classes subset",
    "train_images": TRAIN_IMAGES,
    "val_images": VAL_IMAGES,
    "test_images": TEST_IMAGES,
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "image_size": IMAGE_SIZE,
    "mAP@0.5_PascalVOC": mAP50,
    "mAP@0.5:0.95_COCO": mAP5095,
    "precision": precision,
    "recall": recall,
    "avg_inference_time_seconds_per_image": avg_inference_time,
    "fps": fps,
    "total_parameters": total_parameters,
    "trainable_parameters": trainable_parameters,
    "model_size_MB": model_size_MB,
    "device": DEVICE_NAME,
    "gpu": GPU_NAME
}

pd.DataFrame([evaluation_metrics]).to_csv(
    METRICS_DIR / "evaluation_metrics.csv",
    index=False
)

with open(METRICS_DIR / "evaluation_metrics.json", "w", encoding="utf-8") as f:
    json.dump(evaluation_metrics, f, ensure_ascii=False, indent=2)

print("Saved evaluation metrics.")

Saved evaluation metrics.


#### **6.2. Tự động tạo báo cáo README.md**

In [17]:
readme_content = f'''# YOLOv8s Full Dataset Model for Python Object Detection App

## Purpose

This model is trained on the full VOC0712 traffic 5-class dataset and is intended for deployment in a Python object detection application.

## Classes

- person
- car
- bus
- bicycle
- motorbike

## App Usage

```python
from ultralytics import YOLO

model = YOLO("model/best.pt")
results = model.predict("image.jpg", conf=0.25, iou=0.45, imgsz=640)
```

## Output Files

- `model/best.pt`: main model checkpoint for app deployment
- `model/last.pt`: last training checkpoint
- `config/data.yaml`: dataset config
- `config/training_config.json`: training settings
- `config/class_names.json`: class id to class name mapping
- `config/inference_config.json`: default inference settings
- `metrics/evaluation_metrics.csv`: main evaluation metrics
- `metrics/evaluation_metrics.json`: main evaluation metrics in JSON
- `metrics/per_class_metrics.csv`: per-class mAP
- `predictions/sample_predictions/`: visual prediction examples
- `predictions/prediction_examples.csv`: prediction CSV examples
- `logs/training_log.txt`: training log
- `logs/results.csv`: Ultralytics training history

## Main Metrics

- mAP@0.5: {mAP50}
- mAP@0.5:0.95: {mAP5095}
- Precision: {precision}
- Recall: {recall}
- FPS: {fps}
- Model size MB: {model_size_MB}
'''

with open(MODEL_RESULT_DIR / "README.md", "w", encoding="utf-8") as f:
    f.write(readme_content)

print("Saved README:", MODEL_RESULT_DIR / "README.md")

Saved README: /content/app_model_results/yolov8s_subset/README.md


#### **6.3. Nén toàn bộ thư mục kết quả (.zip)**

In [18]:
ZIP_OUTPUT_PATH = Path("/content/yolov8s_full_dataset_app_model.zip")

if ZIP_OUTPUT_PATH.exists():
    ZIP_OUTPUT_PATH.unlink()

shutil.make_archive(
    base_name=str(ZIP_OUTPUT_PATH).replace(".zip", ""),
    format="zip",
    root_dir=str(OUTPUT_ROOT),
    base_dir=MODEL_NAME
)

print("Created:", ZIP_OUTPUT_PATH)

Created: /content/yolov8s_full_dataset_app_model.zip


#### **6.4. Copy zip lên Drive**

In [19]:
DRIVE_OUTPUT_PATH = Path("/content/drive/MyDrive/ObjectDetection/yolov8s_full_dataset_app_model.zip")

DRIVE_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

shutil.copy(ZIP_OUTPUT_PATH, DRIVE_OUTPUT_PATH)

print("Copied to:", DRIVE_OUTPUT_PATH)

Copied to: /content/drive/MyDrive/ObjectDetection/yolov8s_full_dataset_app_model.zip


#### **6.5. Kiểm tra lại các file đầu ra (Final Check)**

Trước khi gửi file zip, kiểm tra:

In [20]:
!find /content/app_model_results/yolov8s_full_dataset -maxdepth 4 -type f | sort

find: ‘/content/app_model_results/yolov8s_full_dataset’: No such file or directory
